# ✦ Spell Vision × Wan VACE — photoreal wand insertion

Takes the ZIP exported by **`export.html`** (aligned `frames/` + `masks/`) plus a
**wand reference image**, and uses [Wan 2.1 VACE](https://huggingface.co/Wan-AI/Wan2.1-VACE-1.3B-diffusers)
masked video editing to synthesize the wand into the clip with real lighting,
shadows, and occlusion. White mask pixels are regenerated; black pixels keep the
original video.

**Before running:** `Runtime → Change runtime type → GPU`.
- **T4 (free):** works with the 1.3B model + CPU offload, but slow (~30–60 min for 81 frames). Consider `NUM_INFERENCE_STEPS = 25` and trimming to 49 frames.
- **L4 / A100:** comfortable; a few minutes.

**Wand reference image tip:** a photo/render of the wand on a clean **white background**, filling most of the image, works best.

In [ ]:
# 1 · Install (takes ~2 min)
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg opencv-python-headless bitsandbytes

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable a GPU runtime!')

In [ ]:
# 2 · Upload inputs: the export ZIP and one wand reference image.
#     You can upload them together (Ctrl+click to multi-select) or one at a
#     time — the cell keeps asking until it has both. Files already uploaded
#     in a previous run are picked up automatically.
#     (For big files, mount Drive instead and set ZIP_PATH / REF_PATH manually.)
import glob, os
from google.colab import files

ZIP_PATH = None   # e.g. '/content/drive/MyDrive/spellvision_vace_123.zip'
REF_PATH = None   # e.g. '/content/drive/MyDrive/wand_ref.png'

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.webp')

def _scan_content():
    global ZIP_PATH, REF_PATH
    for p in sorted(glob.glob('/content/*'), key=os.path.getmtime, reverse=True):
        if ZIP_PATH is None and p.lower().endswith('.zip'):
            ZIP_PATH = p
        elif REF_PATH is None and p.lower().endswith(IMG_EXTS):
            REF_PATH = p

_scan_content()  # reuse files from an earlier upload/run

while ZIP_PATH is None or REF_PATH is None:
    need = []
    if ZIP_PATH is None: need.append('the ZIP from export.html')
    if REF_PATH is None: need.append('a wand reference image (png/jpg)')
    print('Upload ' + ' AND '.join(need) + ':')
    files.upload()
    _scan_content()

print('ZIP:', ZIP_PATH)
print('Reference:', REF_PATH)

In [ ]:
# 3 · Preprocess: unzip, center-crop + resize frames & masks identically to a
#     Wan-friendly bucket, dilate masks a little (blending margin), trim to 4k+1.
import zipfile, json, glob, os
import numpy as np
import cv2
from PIL import Image

MAX_FRAMES = 81          # Wan wants 4k+1; 81 ≈ 5 s @ 16 fps
MASK_DILATE_FRAC = 0.018 # dilation radius as a fraction of width. Enough that the
                         #   mask overlaps back into the fist edge (the export's
                         #   hand occluder is slightly conservative); raise toward
                         #   0.025 if the wand still floats off the hand.

!rm -rf /content/vace_in && mkdir -p /content/vace_in
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/vace_in')

meta = json.load(open('/content/vace_in/meta.json'))
print('meta:', meta)
frame_paths = sorted(glob.glob('/content/vace_in/frames/*.jpg'))
mask_paths = sorted(glob.glob('/content/vace_in/masks/*.png'))
assert len(frame_paths) == len(mask_paths) and frame_paths, 'frames/masks mismatch'

n = min(len(frame_paths), MAX_FRAMES)
n = (n - 1) // 4 * 4 + 1  # → 4k+1
frame_paths, mask_paths = frame_paths[:n], mask_paths[:n]

# Pick the bucket (all multiples of 16) closest to the source aspect.
src_w, src_h = meta['width'], meta['height']
buckets = [(832, 480), (480, 832), (624, 624), (704, 544), (544, 704)]
W, H = min(buckets, key=lambda wh: abs(wh[0] / wh[1] - src_w / src_h))
print(f'{n} frames, {src_w}x{src_h} → {W}x{H}')

def crop_resize(img, w, h, resample):
    # Cover-crop to the target aspect, then resize — same transform for frames
    # and masks keeps them pixel-aligned.
    sw, sh = img.size
    scale = max(w / sw, h / sh)
    nw, nh = round(sw * scale), round(sh * scale)
    img = img.resize((nw, nh), resample)
    left, top = (nw - w) // 2, (nh - h) // 2
    return img.crop((left, top, left + w, top + h))

kernel_r = max(1, int(MASK_DILATE_FRAC * W))
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * kernel_r + 1, 2 * kernel_r + 1))

video_frames, mask_frames = [], []
for fp, mp in zip(frame_paths, mask_paths):
    video_frames.append(crop_resize(Image.open(fp).convert('RGB'), W, H, Image.LANCZOS))
    m = crop_resize(Image.open(mp).convert('L'), W, H, Image.BILINEAR)
    m = (np.asarray(m) > 96).astype(np.uint8) * 255
    m = cv2.dilate(m, kernel)
    mask_frames.append(Image.fromarray(m))

ref_image = crop_resize(Image.open(REF_PATH).convert('RGB'), W, H, Image.LANCZOS)

# Sanity preview: original / mask overlay / reference.
import matplotlib.pyplot as plt
idx = int(np.argmax([np.asarray(m).sum() for m in mask_frames]))  # frame with the most wand
overlay = np.asarray(video_frames[idx]).copy()
overlay[np.asarray(mask_frames[idx]) > 0] = [255, 40, 40]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, im, title in zip(axes, [video_frames[idx], overlay, ref_image],
                          [f'frame {idx}', 'mask overlay (red = regenerate)', 'wand reference']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.show()
print('If the red region is NOT a wand held in the hand, fix the export before burning GPU time.')

In [ ]:
# 4 · Prompts + two-stage model load (free-Colab RAM safe).
#
# The UMT5-XXL text encoder (~11 GB fp16) cannot coexist with the rest of the
# pipeline in Colab's ~12.7 GB system RAM. So it is loaded ALONE (4-bit, on
# GPU), used once to turn the prompts into embeddings, then deleted — and the
# pipeline is loaded WITHOUT a text encoder, consuming the embeddings directly.
#
# Edit the prompt to match YOUR wand reference, then re-run this cell.
PROMPT = (
    'A person holds a slender dark wooden magic wand in their hand, gripping it '
    'naturally in a fist. The wand is a real physical object: it matches the '
    'scene lighting and white balance, has natural soft shadows and correct '
    'perspective, and stays rigidly in the hand as it moves. Photorealistic, '
    'natural indoor lighting, realistic skin, sharp details.'
)
NEGATIVE = (
    'Bright tones, overexposed, static, blurred details, subtitles, style, works, '
    'paintings, images, static, overall gray, worst quality, low quality, JPEG '
    'compression residue, ugly, incomplete, extra fingers, poorly drawn hands, '
    'poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, '
    'still picture, messy background, cartoon, CGI, render, floating object'
)

import gc
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, UMT5EncoderModel
from diffusers import AutoencoderKLWan, WanVACEPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

MODEL_ID = 'Wan-AI/Wan2.1-VACE-1.3B-diffusers'  # 14B variant needs A100-80GB

# T4 (compute 7.5) has no usable bfloat16 — fall back to float16 there.
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

# -- Stage 1: text encoder alone → prompt embeddings --------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder='tokenizer')
text_encoder = UMT5EncoderModel.from_pretrained(
    MODEL_ID, subfolder='text_encoder', torch_dtype=dtype,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=dtype,
    ),
    device_map='cuda',
)

def encode(text, max_len=512):
    # Mirrors Wan's encode_prompt: encode, then zero the padding positions.
    ins = tokenizer(
        text, padding='max_length', max_length=max_len, truncation=True,
        add_special_tokens=True, return_attention_mask=True, return_tensors='pt',
    ).to('cuda')
    with torch.no_grad():
        emb = text_encoder(ins.input_ids, ins.attention_mask).last_hidden_state
    n = int(ins.attention_mask[0].sum())
    emb[0, n:] = 0
    return emb.to(dtype).cpu()

prompt_embeds = encode(PROMPT)
negative_prompt_embeds = encode(NEGATIVE)
del text_encoder
gc.collect(); torch.cuda.empty_cache()
print('prompts encoded, text encoder freed')

# -- Stage 2: the rest of the pipeline, no text encoder -----------------------
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(
    MODEL_ID, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=dtype,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)  # 3.0 for 480p
pipe.enable_model_cpu_offload()  # fits in 16 GB VRAM; remove on A100 for speed
if hasattr(pipe.vae, 'enable_tiling'):
    pipe.vae.enable_tiling()
gc.collect(); torch.cuda.empty_cache()
print('pipeline ready,', dtype)

In [ ]:
# 5 · Generate. (Prompts live in cell 4 — edit there and re-run it first.)
NUM_INFERENCE_STEPS = 30   # 25 is noticeably faster on T4, slightly softer
SEED = 42

output = pipe(
    video=video_frames,
    mask=mask_frames,
    reference_images=[ref_image],
    prompt_embeds=prompt_embeds.to('cuda'),
    negative_prompt_embeds=negative_prompt_embeds.to('cuda'),
    height=H,
    width=W,
    num_frames=len(video_frames),
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=5.0,
    generator=torch.Generator().manual_seed(SEED),
).frames[0]
print('done:', len(output), 'frames')

In [ ]:
# 6 · Export result + side-by-side comparison, then download.
import numpy as np
from PIL import Image
from diffusers.utils import export_to_video
from google.colab import files

def to_np(im):
    a = np.asarray(im)
    if a.dtype != np.uint8:  # pipeline returns float arrays in [0, 1]
        a = (a * 255).round().clip(0, 255).astype(np.uint8)
    return a

result = [Image.fromarray(to_np(f)) for f in output]
FPS = meta.get('fps', 16)
export_to_video(result, '/content/wand_result.mp4', fps=FPS)

side = [np.concatenate([to_np(a), to_np(b)], axis=1) for a, b in zip(video_frames, result)]
export_to_video([Image.fromarray(f) for f in side], '/content/wand_compare.mp4', fps=FPS)

files.download('/content/wand_result.mp4')
files.download('/content/wand_compare.mp4')

## If the result is off
- **Wand ignored / wrong object** → better reference image (white background, wand fills the frame), and describe it precisely in `PROMPT` (cell 4).
- **Wand doesn't reach the fingers / floats** → the mask is too tight; raise `MASK_DILATE_FRAC` (e.g. `0.025`).
- **Hand gets repainted / warped** → mask too loose; lower `MASK_DILATE_FRAC`, and check the red overlay in step 3 stays off the fingers.
- **Flicker** → more steps (40–50), or try a different `SEED`.
- **Session crashed (system RAM)** → restart runtime, rerun cells 1→4. Cell 4 must run as-is: the text encoder is loaded alone in 4-bit, used, and deleted *before* the pipeline loads — never load both together on free Colab. Already-uploaded files are picked up automatically.
- **CUDA out of memory on T4** → trim to 49 frames in step 3 (`MAX_FRAMES = 49`) — never skip `enable_model_cpu_offload()`.
- **Quality ceiling** → same notebook with `MODEL_ID = 'Wan-AI/Wan2.1-VACE-14B-diffusers'` on an A100 runtime, `flow_shift=5.0` if you also bump to 720p buckets.